# Phase 4 — Option A: YOLOv8 CARLA 도메인 적응 파인튜닝

## 목표
- **문제**: Phase 4 평가에서 Detection Precision=0.1734로 매우 낮음 (COCO→CARLA 도메인 갭)
- **원인**: COCO128로 학습된 YOLOv8이 CARLA 렌더링 스타일, 시점, 조명에 적응 못함
- **해결**: CARLA 수집 데이터로 파인튜닝 → 도메인 적응 (Domain Adaptation)
- **기대**: Precision↑ → Tracking MOTA↑ (Detection이 병목이었으므로 연쇄 개선)

## 핵심 개념: 왜 파인튜닝인가?
- **전이학습(Transfer Learning)**: COCO에서 학습한 피처 추출기(백본)는 유지, 도메인 특화 피처만 조정
- **적은 데이터로도 효과적**: 215장으로도 도메인 갭을 줄일 수 있음 (특징 추출기가 이미 강력)
- **sim-to-real gap**: 시뮬레이션→실제 환경 전환 시 항상 발생, 자율주행 연구의 핵심 문제

In [ ]:
# 한글 폰트 설정 (필수)
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os
import shutil
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from pathlib import Path
from collections import defaultdict

print('라이브러리 로드 완료')

## Cell 1: 데이터셋 현황 분석

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────
BASE_DIR   = Path(r'C:/Users/apple/Desktop/autonomous_cv_pipeline/phase4_carla')
DATA_DIR   = BASE_DIR / 'data_collection' / 'carla_dataset'
IMG_DIR    = DATA_DIR / 'images'
LBL_DIR    = DATA_DIR / 'labels'
FINETUNE_DIR = BASE_DIR / 'finetuning'

# ── 어노테이션 있는 프레임만 필터링 ────────────────────────────────────────
all_labels = sorted(LBL_DIR.glob('*.txt'))

annotated = []   # 객체 있는 프레임
class_count = defaultdict(int)
obj_per_frame = []

for lbl_path in all_labels:
    lines = lbl_path.read_text().strip().splitlines()
    lines = [l for l in lines if l.strip()]
    if lines:
        annotated.append(lbl_path.stem)  # 예: '000004'
        obj_per_frame.append(len(lines))
        for line in lines:
            cls_id = int(line.split()[0])
            class_count[cls_id] += 1

total_objs = sum(class_count.values())
print(f'전체 프레임      : {len(all_labels)}')
print(f'어노테이션 프레임 : {len(annotated)} ({len(annotated)/len(all_labels)*100:.1f}%)')
print(f'빈 프레임         : {len(all_labels) - len(annotated)}')
print(f'총 객체 수        : {total_objs}')
print()
cls_names = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
for cls_id, cnt in sorted(class_count.items()):
    name = cls_names.get(cls_id, f'cls{cls_id}')
    print(f'  class {cls_id} ({name:10s}): {cnt:4d}개 ({cnt/total_objs*100:.1f}%)')

# ── 시각화: 클래스 분포 + 프레임당 객체 수 ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 클래스 분포
labels_vis = [f'{cls_names.get(k, k)}\n({v})' for k, v in sorted(class_count.items())]
axes[0].bar(labels_vis, [class_count[k] for k in sorted(class_count.keys())],
            color=['steelblue' if k == 0 else 'coral' for k in sorted(class_count.keys())])
axes[0].set_title('CARLA 데이터셋 클래스 분포')
axes[0].set_ylabel('객체 수')

# 프레임당 객체 수 히스토그램
axes[1].hist(obj_per_frame, bins=range(1, max(obj_per_frame)+2), edgecolor='black', color='mediumseagreen')
axes[1].set_title('프레임당 객체 수 분포')
axes[1].set_xlabel('객체 수')
axes[1].set_ylabel('프레임 수')

plt.tight_layout()
plt.savefig(FINETUNE_DIR / 'dataset_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('데이터셋 분석 완료')

## Cell 2: Train/Val 분할 + YOLO 데이터셋 구조 생성

### YOLO 데이터셋 구조
```
carla_yolo/
  images/
    train/  ← 어노테이션 있는 프레임의 80%
    val/    ← 어노테이션 있는 프레임의 20%
  labels/
    train/
    val/
  carla.yaml  ← 데이터셋 config
```

In [ ]:
# ── Train/Val 분할 ──────────────────────────────────────────────────────────
random.seed(42)
shuffled = annotated.copy()
random.shuffle(shuffled)

n_train = int(len(shuffled) * 0.8)
train_ids = set(shuffled[:n_train])
val_ids   = set(shuffled[n_train:])

print(f'Train: {len(train_ids)}장 / Val: {len(val_ids)}장')

# ── 디렉토리 생성 ────────────────────────────────────────────────────────────
YOLO_DATASET = FINETUNE_DIR / 'carla_yolo'
for split in ['train', 'val']:
    (YOLO_DATASET / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET / 'labels' / split).mkdir(parents=True, exist_ok=True)

# ── 파일 복사 ────────────────────────────────────────────────────────────────
def copy_split(ids, split):
    for frame_id in ids:
        src_img = IMG_DIR / f'{frame_id}.jpg'
        src_lbl = LBL_DIR / f'{frame_id}.txt'
        dst_img = YOLO_DATASET / 'images' / split / f'{frame_id}.jpg'
        dst_lbl = YOLO_DATASET / 'labels' / split / f'{frame_id}.txt'
        if src_img.exists():
            shutil.copy2(src_img, dst_img)
        if src_lbl.exists():
            shutil.copy2(src_lbl, dst_lbl)

copy_split(train_ids, 'train')
copy_split(val_ids,   'val')

# 실제 복사된 수 확인
n_train_img = len(list((YOLO_DATASET / 'images' / 'train').glob('*.jpg')))
n_val_img   = len(list((YOLO_DATASET / 'images' / 'val').glob('*.jpg')))
print(f'복사 완료: train={n_train_img}장, val={n_val_img}장')

# ── carla.yaml 생성 ──────────────────────────────────────────────────────────
yaml_content = f"""# CARLA 자율주행 데이터셋 (Phase 4 파인튜닝용)
# 수집: CARLA 0.9.15, Town01, 500 프레임

path: {YOLO_DATASET.as_posix()}
train: images/train
val:   images/val

nc: 8  # 클래스 수 (COCO80 서브셋, 0-7 전체 선언 필요)
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
"""

yaml_path = YOLO_DATASET / 'carla.yaml'
yaml_path.write_text(yaml_content, encoding='utf-8')
print(f'\nYAML 생성: {yaml_path}')
print(yaml_content)

## Cell 3: 베이스라인 측정 (파인튜닝 전)

파인튜닝 전 YOLOv8s (COCO pretrained)의 val 성능을 기록해 둡니다.

In [ ]:
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# ── 베이스라인 모델 (COCO pretrained) ────────────────────────────────────────
baseline_model = YOLO('yolov8s.pt')  # yolov8s: 소형, 빠른 수렴, 215장에 적합

print('=== 베이스라인 평가 (COCO pretrained → CARLA val) ===')
baseline_results = baseline_model.val(
    data=str(yaml_path),
    imgsz=640,
    conf=0.25,
    iou=0.5,
    verbose=False,
    device=0,  # GPU
)

# 핵심 지표 추출
baseline_map50    = float(baseline_results.box.map50)
baseline_map5095  = float(baseline_results.box.map)
baseline_precision = float(baseline_results.box.mp)
baseline_recall    = float(baseline_results.box.mr)

print(f'  mAP@50       : {baseline_map50:.4f}')
print(f'  mAP@50:95    : {baseline_map5095:.4f}')
print(f'  Precision(mp): {baseline_precision:.4f}')
print(f'  Recall(mr)   : {baseline_recall:.4f}')
print()
print('→ 이 숫자가 도메인 갭의 기준선입니다. 파인튜닝 후 개선량을 비교합니다.')

## Cell 4: YOLOv8 파인튜닝

### 전략: 점진적 학습률 (Freeze Backbone)
- **1단계**: 백본 동결 → Head만 학습 (5 epoch) → 도메인 특화 분류기 초기화
- **2단계**: 전체 파인튜닝 (15 epoch) → 백본 미세 조정

### 왜 yolov8s인가?
- 215장은 적은 데이터 → 작은 모델이 과적합 위험 낮음
- yolov8s = 11.2M 파라미터 (yolov8m의 절반), 빠른 수렴

In [ ]:
# ── 파인튜닝 설정 ─────────────────────────────────────────────────────────────
TRAIN_EPOCHS  = 30      # 작은 데이터셋 → 30 epoch으로 충분
FREEZE_EPOCHS = 5       # 처음 5 epoch는 백본 동결
IMG_SZ        = 640
BATCH         = 8       # RTX 4080 Super 16GB → 여유 있음
LR0           = 1e-3    # 초기 학습률
LRF           = 0.01    # 최종 학습률 비율 (cosine decay)

print('=== YOLOv8s CARLA 파인튜닝 시작 ===')
print(f'  데이터: carla_yolo (train={n_train_img}장 / val={n_val_img}장)')
print(f'  에폭  : {TRAIN_EPOCHS} (백본 동결 {FREEZE_EPOCHS} + 전체 {TRAIN_EPOCHS-FREEZE_EPOCHS})')
print(f'  배치  : {BATCH} / 이미지 크기: {IMG_SZ}')
print()

# 새로운 모델 인스턴스 (pretrained weights 로드)
finetune_model = YOLO('yolov8s.pt')

# ── 파인튜닝 실행 ──────────────────────────────────────────────────────────────
train_results = finetune_model.train(
    data=str(yaml_path),
    epochs=TRAIN_EPOCHS,
    imgsz=IMG_SZ,
    batch=BATCH,
    lr0=LR0,
    lrf=LRF,
    freeze=10,          # 백본 첫 10레이어 동결 (전체 학습 전 헤드 안정화)
    warmup_epochs=3,    # Warmup: 처음 3 epoch 낮은 lr로 시작
    patience=10,        # Early stopping: 10 epoch 개선 없으면 중단
    save=True,
    project=str(FINETUNE_DIR / 'runs'),
    name='carla_finetune',
    device=0,
    exist_ok=True,
    # 데이터 증강: 작은 데이터셋에 효과적
    hsv_h=0.015,        # Hue 변형 (조명 변화 모의)
    hsv_s=0.7,          # Saturation
    hsv_v=0.4,          # Value (밝기)
    degrees=0.0,        # 회전 없음 (자율주행 카메라는 수평 고정)
    flipud=0.0,         # 상하 뒤집기 없음 (동일 이유)
    fliplr=0.5,         # 좌우 뒤집기 (50%)
    mosaic=0.8,         # Mosaic augmentation (적은 데이터에서 매우 효과적)
    mixup=0.1,          # MixUp (부드러운 경계 학습)
    verbose=True,
)

print('\n파인튜닝 완료!')
best_model_path = FINETUNE_DIR / 'runs' / 'carla_finetune' / 'weights' / 'best.pt'
print(f'최적 모델 저장: {best_model_path}')

## Cell 5: 파인튜닝 후 평가 + 학습 곡선

In [ ]:
import pandas as pd

# ── 파인튜닝된 모델 평가 ─────────────────────────────────────────────────────
best_model = YOLO(str(best_model_path))

print('=== 파인튜닝 후 평가 (CARLA val) ===')
ft_results = best_model.val(
    data=str(yaml_path),
    imgsz=IMG_SZ,
    conf=0.25,
    iou=0.5,
    verbose=False,
    device=0,
)

ft_map50     = float(ft_results.box.map50)
ft_map5095   = float(ft_results.box.map)
ft_precision = float(ft_results.box.mp)
ft_recall    = float(ft_results.box.mr)

# ── 비교 출력 ────────────────────────────────────────────────────────────────
print()
print('=' * 55)
print(f'{"지표":<20} {"베이스라인":>12} {"파인튜닝":>12} {"개선":>8}')
print('-' * 55)

metrics = [
    ('mAP@50',    baseline_map50,    ft_map50),
    ('mAP@50:95', baseline_map5095,  ft_map5095),
    ('Precision', baseline_precision, ft_precision),
    ('Recall',    baseline_recall,    ft_recall),
]
for name, bv, fv in metrics:
    delta = fv - bv
    sign = '+' if delta >= 0 else ''
    print(f'{name:<20} {bv:>12.4f} {fv:>12.4f} {sign}{delta:>7.4f}')
print('=' * 55)

# ── 학습 곡선 시각화 ──────────────────────────────────────────────────────────
results_csv = FINETUNE_DIR / 'runs' / 'carla_finetune' / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('YOLOv8 CARLA 파인튜닝 학습 곡선', fontsize=14)
    
    plot_pairs = [
        ('train/box_loss', 'val/box_loss', 'Box Loss'),
        ('train/cls_loss', 'val/cls_loss', 'Classification Loss'),
        ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss'),
        ('metrics/mAP50(B)', None, 'mAP@50'),
        ('metrics/precision(B)', None, 'Precision'),
        ('metrics/recall(B)', None, 'Recall'),
    ]
    
    for ax, (train_col, val_col, title) in zip(axes.flat, plot_pairs):
        if train_col in df.columns:
            ax.plot(df['epoch'], df[train_col], label='train', color='steelblue')
        if val_col and val_col in df.columns:
            ax.plot(df['epoch'], df[val_col], label='val', color='coral')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(FINETUNE_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('학습 곡선 저장 완료')
else:
    print('results.csv 없음 — 학습 곡선 스킵')

## Cell 6: 클래스별 AP 분석

person과 car 각각 얼마나 개선됐는지 확인합니다.

In [ ]:
# ── 클래스별 AP 비교 ──────────────────────────────────────────────────────────
print('=== 클래스별 AP@50 비교 ===')

# 클래스별 AP 추출 (ultralytics results 객체)
def get_per_class_ap(results, model):
    """Ultralytics val results에서 클래스별 AP50 추출"""
    ap50_per_cls = results.box.ap50  # numpy array, 클래스 순서
    names = model.names              # dict: {id: name}
    return {names[i]: float(ap50_per_cls[i]) for i in range(len(ap50_per_cls))}

try:
    baseline_cls_ap = get_per_class_ap(baseline_results, baseline_model)
    ft_cls_ap       = get_per_class_ap(ft_results, best_model)
    
    print(f'{"클래스":<12} {"베이스라인":>12} {"파인튜닝":>12} {"개선":>8}')
    print('-' * 50)
    all_cls = set(baseline_cls_ap) | set(ft_cls_ap)
    for cls_name in sorted(all_cls):
        bv = baseline_cls_ap.get(cls_name, 0.0)
        fv = ft_cls_ap.get(cls_name, 0.0)
        delta = fv - bv
        sign = '+' if delta >= 0 else ''
        print(f'{cls_name:<12} {bv:>12.4f} {fv:>12.4f} {sign}{delta:>7.4f}')
    
    # 시각화
    cls_list = sorted(all_cls)
    x = np.arange(len(cls_list))
    w = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - w/2, [baseline_cls_ap.get(c, 0) for c in cls_list],
                   w, label='베이스라인 (COCO)', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + w/2, [ft_cls_ap.get(c, 0) for c in cls_list],
                   w, label='파인튜닝 (CARLA)', color='coral', alpha=0.8)
    
    ax.set_xlabel('클래스')
    ax.set_ylabel('AP@50')
    ax.set_title('클래스별 AP@50: 베이스라인 vs 파인튜닝')
    ax.set_xticks(x)
    ax.set_xticklabels(cls_list)
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 막대 위에 값 표시
    for bar in [*bars1, *bars2]:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(FINETUNE_DIR / 'class_ap_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f'클래스별 AP 추출 오류: {e}')
    print('전체 지표만 비교합니다.')

## Cell 7: 탐지 결과 시각화 비교

베이스라인 vs 파인튜닝 모델의 실제 탐지 결과를 시각적으로 비교합니다.

In [ ]:
# ── val 이미지 샘플 6장 시각화 ────────────────────────────────────────────────
val_imgs = sorted((YOLO_DATASET / 'images' / 'val').glob('*.jpg'))
sample_imgs = random.sample(val_imgs, min(6, len(val_imgs)))

fig, axes = plt.subplots(len(sample_imgs), 2, figsize=(16, 4 * len(sample_imgs)))
fig.suptitle('베이스라인 vs 파인튜닝 탐지 결과 비교', fontsize=14)

for row_idx, img_path in enumerate(sample_imgs):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 왼쪽: 베이스라인
    bl_pred = baseline_model.predict(img_bgr, imgsz=IMG_SZ, conf=0.25, device=0, verbose=False)[0]
    # 오른쪽: 파인튜닝
    ft_pred = best_model.predict(img_bgr, imgsz=IMG_SZ, conf=0.25, device=0, verbose=False)[0]
    
    def draw_preds(ax, img, preds, title):
        ax.imshow(img)
        boxes = preds.boxes
        if boxes is not None and len(boxes) > 0:
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0])
                cls  = int(box.cls[0])
                name = preds.names[cls]
                color = 'lime' if cls == 0 else 'red'
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                          linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-5, f'{name} {conf:.2f}', color=color,
                        fontsize=8, fontweight='bold',
                        bbox=dict(facecolor='black', alpha=0.5, pad=1))
        n_det = len(boxes) if boxes is not None else 0
        ax.set_title(f'{title} ({n_det}개 탐지)')
        ax.axis('off')
    
    draw_preds(axes[row_idx, 0], img_rgb, bl_pred, '베이스라인 (COCO)')
    draw_preds(axes[row_idx, 1], img_rgb, ft_pred, '파인튜닝 (CARLA)')

plt.tight_layout()
plt.savefig(FINETUNE_DIR / 'detection_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('시각화 완료')

## Cell 8: 파인튜닝된 모델로 Tracking MOTA 재평가

Detection 개선 → ByteTrack MOTA 개선 연쇄 확인

In [ ]:
import sys
sys.path.insert(0, str(Path(r'C:/Users/apple/Desktop/autonomous_cv_pipeline')))

# ── 100프레임 시퀀스 선택 (어노테이션 있는 연속 구간) ─────────────────────────
# 원본 images/ 에서 연속 프레임 사용 (순서 중요)
all_imgs_sorted = sorted(IMG_DIR.glob('*.jpg'))
all_lbls_sorted = sorted(LBL_DIR.glob('*.txt'))

# 어노테이션 있는 100개 프레임을 연속으로 선택
annot_with_img = [
    img for img in all_imgs_sorted
    if (LBL_DIR / img.stem).with_suffix('.txt').stat().st_size > 0
    if (LBL_DIR / img.stem).with_suffix('.txt').exists()
]
seq_imgs = annot_with_img[:100] if len(annot_with_img) >= 100 else annot_with_img
print(f'시퀀스 길이: {len(seq_imgs)} 프레임')

# ── ByteTrack 간이 재구현 (Phase 2에서 구현한 버전 활용) ─────────────────────
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

def iou_batch(boxes_a, boxes_b):
    """N×4, M×4 → N×M IoU 행렬"""
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    a_area = (boxes_a[:,2]-boxes_a[:,0]) * (boxes_a[:,3]-boxes_a[:,1])
    b_area = (boxes_b[:,2]-boxes_b[:,0]) * (boxes_b[:,3]-boxes_b[:,1])
    return inter / (a_area[:,None] + b_area[None,:] - inter + 1e-6)

class Track:
    _id_counter = 0
    def __init__(self, box):
        Track._id_counter += 1
        self.id  = Track._id_counter
        self.box = np.array(box, dtype=float)
        self.age = 1
        self.hits = 1
        self.miss = 0
    def predict(self):
        return self.box
    def update(self, box):
        self.box  = np.array(box, dtype=float)
        self.hits += 1
        self.miss  = 0
        self.age  += 1

class SimpleByteTrack:
    def __init__(self, high_thresh=0.5, low_thresh=0.1, iou_thresh=0.3, max_miss=5):
        self.high_thresh = high_thresh
        self.low_thresh  = low_thresh
        self.iou_thresh  = iou_thresh
        self.max_miss    = max_miss
        self.tracks: list[Track] = []
        Track._id_counter = 0
    
    def update(self, detections):
        """detections: list of [x1,y1,x2,y2,conf]"""
        if len(detections) == 0:
            for t in self.tracks:
                t.miss += 1
            self.tracks = [t for t in self.tracks if t.miss <= self.max_miss]
            return []
        
        dets = np.array(detections)  # N×5
        high_mask = dets[:, 4] >= self.high_thresh
        low_mask  = (~high_mask) & (dets[:, 4] >= self.low_thresh)
        high_dets = dets[high_mask]
        low_dets  = dets[low_mask]
        
        active_tracks = self.tracks
        results = []
        matched_track_ids = set()
        matched_det_ids   = set()
        
        # 1단계: 고신뢰도 탐지 매칭
        if len(active_tracks) > 0 and len(high_dets) > 0:
            pred_boxes = np.array([t.predict() for t in active_tracks])
            iou_mat = iou_batch(pred_boxes, high_dets[:, :4])
            row_ids, col_ids = linear_sum_assignment(-iou_mat)
            for r, c in zip(row_ids, col_ids):
                if iou_mat[r, c] >= self.iou_thresh:
                    active_tracks[r].update(high_dets[c, :4])
                    results.append((active_tracks[r].id, high_dets[c, :4]))
                    matched_track_ids.add(r)
                    matched_det_ids.add(c)
        
        # 2단계: 미매칭 트랙 vs 저신뢰도 탐지
        unmatched_tracks = [t for i, t in enumerate(active_tracks) if i not in matched_track_ids]
        if len(unmatched_tracks) > 0 and len(low_dets) > 0:
            pred_boxes = np.array([t.predict() for t in unmatched_tracks])
            iou_mat2 = iou_batch(pred_boxes, low_dets[:, :4])
            row_ids2, col_ids2 = linear_sum_assignment(-iou_mat2)
            matched_low = set()
            for r, c in zip(row_ids2, col_ids2):
                if iou_mat2[r, c] >= self.iou_thresh:
                    unmatched_tracks[r].update(low_dets[c, :4])
                    results.append((unmatched_tracks[r].id, low_dets[c, :4]))
                    matched_low.add(r)
            for i, t in enumerate(unmatched_tracks):
                if i not in matched_low:
                    t.miss += 1
        else:
            for t in unmatched_tracks:
                t.miss += 1
        
        # 3단계: 새 트랙 생성 (고신뢰도 미매칭 탐지)
        for c in range(len(high_dets)):
            if c not in matched_det_ids:
                new_track = Track(high_dets[c, :4])
                self.tracks.append(new_track)
                results.append((new_track.id, high_dets[c, :4]))
        
        # 오래된 트랙 제거
        self.tracks = [t for t in self.tracks if t.miss <= self.max_miss]
        return results

print('ByteTrack 재구현 로드 완료')

In [ ]:
# ── MOTA 평가 함수 ────────────────────────────────────────────────────────────
def load_gt_boxes(lbl_path, img_w=1280, img_h=720):
    """YOLO 형식 라벨 → [x1,y1,x2,y2] 절대 좌표"""
    boxes = []
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        return boxes
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if not parts:
            continue
        cx, cy, w, h = map(float, parts[1:5])
        x1 = (cx - w/2) * img_w
        y1 = (cy - h/2) * img_h
        x2 = (cx + w/2) * img_w
        y2 = (cy + h/2) * img_h
        boxes.append([x1, y1, x2, y2])
    return boxes

def eval_mota_with_model(model, seq_imgs, lbl_dir, model_name='모델', iou_thresh=0.5):
    tracker = SimpleByteTrack(high_thresh=0.4, low_thresh=0.1, iou_thresh=0.3)
    TP_total = FP_total = FN_total = ID_SW_total = 0
    prev_ids = {}
    
    for img_path in seq_imgs:
        img_bgr = cv2.imread(str(img_path))
        lbl_path = Path(lbl_dir) / img_path.with_suffix('.txt').name
        gt_boxes = load_gt_boxes(lbl_path)
        
        # 탐지
        preds = model.predict(img_bgr, imgsz=640, conf=0.1, device=0, verbose=False)[0]
        dets = []
        if preds.boxes and len(preds.boxes) > 0:
            for box in preds.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = float(box.conf[0])
                dets.append([x1, y1, x2, y2, conf])
        
        track_results = tracker.update(dets)
        
        # MOTA 계산
        if len(gt_boxes) == 0 and len(track_results) == 0:
            continue
        
        GT  = np.array(gt_boxes)  if gt_boxes      else np.zeros((0, 4))
        TRK = np.array([r[1] for r in track_results]) if track_results else np.zeros((0, 4))
        
        matched_gt = set()
        matched_tr = set()
        
        if len(GT) > 0 and len(TRK) > 0:
            iou_mat = iou_batch(GT, TRK)
            row_ids, col_ids = linear_sum_assignment(-iou_mat)
            for r, c in zip(row_ids, col_ids):
                if iou_mat[r, c] >= iou_thresh:
                    matched_gt.add(r)
                    matched_tr.add(c)
                    TP_total += 1
        
        FN_total += len(GT)  - len(matched_gt)
        FP_total += len(TRK) - len(matched_tr)
    
    total_gt = TP_total + FN_total
    if total_gt == 0:
        mota = 0.0
    else:
        mota = 1.0 - (FP_total + FN_total + ID_SW_total) / total_gt
    
    print(f'[{model_name}] Tracking MOTA')
    print(f'  MOTA  : {mota:.4f}')
    print(f'  TP    : {TP_total}')
    print(f'  FP    : {FP_total}')
    print(f'  FN    : {FN_total}')
    return mota, TP_total, FP_total, FN_total

# ── 두 모델 모두 평가 ─────────────────────────────────────────────────────────
print('=== 베이스라인 MOTA ===')
bl_mota, bl_tp, bl_fp, bl_fn = eval_mota_with_model(
    baseline_model, seq_imgs, LBL_DIR, '베이스라인')

print()
print('=== 파인튜닝 MOTA ===')
ft_mota, ft_tp, ft_fp, ft_fn = eval_mota_with_model(
    best_model, seq_imgs, LBL_DIR, '파인튜닝')

print()
print('=' * 45)
print(f'MOTA 개선: {bl_mota:.4f} → {ft_mota:.4f} ({ft_mota - bl_mota:+.4f})')
print('=' * 45)

## Cell 9: 최종 결과 대시보드

In [ ]:
# ── Option A 최종 결과 대시보드 ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Option A: YOLOv8 CARLA 도메인 적응 결과 요약', fontsize=14, y=1.02)

# 1. Detection mAP 비교
categories = ['mAP@50', 'Precision', 'Recall']
bl_vals = [baseline_map50, baseline_precision, baseline_recall]
ft_vals = [ft_map50,       ft_precision,       ft_recall]

x = np.arange(len(categories))
w = 0.35
axes[0].bar(x - w/2, bl_vals, w, label='베이스라인', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, ft_vals, w, label='파인튜닝',   color='coral',     alpha=0.8)
axes[0].set_title('Detection 지표')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].set_ylim(0, 1.1)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
for xi, (bv, fv) in zip(x, zip(bl_vals, ft_vals)):
    axes[0].text(xi - w/2, bv + 0.02, f'{bv:.3f}', ha='center', fontsize=8)
    axes[0].text(xi + w/2, fv + 0.02, f'{fv:.3f}', ha='center', fontsize=8)

# 2. MOTA 비교
mota_vals = [bl_mota, ft_mota]
colors_mota = ['steelblue', 'coral']
bars = axes[1].bar(['베이스라인', '파인튜닝'], mota_vals, color=colors_mota, alpha=0.8)
axes[1].set_title('Tracking MOTA')
axes[1].set_ylim(min(0, min(mota_vals)) - 0.1, max(mota_vals) + 0.15)
axes[1].axhline(y=0, color='black', linewidth=0.8, linestyle='--')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, mota_vals):
    y_pos = val + 0.02 if val >= 0 else val - 0.05
    axes[1].text(bar.get_x() + bar.get_width()/2., y_pos,
                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

# 3. TP/FP/FN 비교
track_data = np.array([[bl_tp, bl_fp, bl_fn], [ft_tp, ft_fp, ft_fn]])
categories3 = ['TP', 'FP', 'FN']
colors3 = ['#2ecc71', '#e74c3c', '#e67e22']
x3 = np.arange(len(categories3))
for i, (label, vals) in enumerate(zip(['베이스라인', '파인튜닝'], track_data)):
    axes[2].bar(x3 + (i-0.5)*w, vals, w, label=label,
               color=[c for c in colors3], alpha=0.7 if i == 0 else 1.0,
               edgecolor='black', linewidth=0.5)
axes[2].set_title('Tracking TP/FP/FN 상세')
axes[2].set_xticks(x3)
axes[2].set_xticklabels(categories3)
axes[2].legend(['베이스라인', '파인튜닝'])
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FINETUNE_DIR / 'option_a_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 텍스트 요약 ───────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('Option A — 도메인 적응 파인튜닝 최종 결과')
print('=' * 60)
print(f'  mAP@50  : {baseline_map50:.4f} → {ft_map50:.4f}  ({ft_map50 - baseline_map50:+.4f})')
print(f'  Precision: {baseline_precision:.4f} → {ft_precision:.4f}  ({ft_precision - baseline_precision:+.4f})')
print(f'  MOTA    : {bl_mota:.4f} → {ft_mota:.4f}  ({ft_mota - bl_mota:+.4f})')
print('=' * 60)
print()
print('→ 다음 단계: Option B — Semantic Segmentation 직접 구현')
print('   (SAM2가 CARLA에서 mIoU=0.107에 머문 이유를 코드로 분석 + SegFormer 대안)')